<a href="https://colab.research.google.com/github/jhhlim/LLMFundamentals/blob/main/Jason_Lim_hw_1a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Class 1a - Introduction to Large Language Models</h1>

# Jason Lim - HW 1a
## Experiment 1: return_full_text
- False = only new text
- True = prompt + new text
## Experiment 2: max_new_tokens
- 500 = long answer
- 20 = shorter
- 5 = very short
## Experiment 3: do_sample
- False = same output every run
- True = different output each run

In [ ]:
print("Hello Class")

Hello Class


In [ ]:
for k in range(10):
  print(k)

0
1
2
3
4
5
6
7
8
9


# This is text call

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
%%capture
!pip install transformers>=4.40.1 accelerate>=0.27.2

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [ ]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=True,
    max_new_tokens=50000,
    do_sample=True
)

For more info on "pipeline" class:

https://huggingface.co/docs/transformers/en/main_classes/pipelines

return_full_text=True (Default for text-generation): The pipeline returns the original input text concatenated with the newly generated text.

return_full_text=False: The pipeline returns only the new, generated text, excluding the input prompt.

The max_new_tokens parameter in the Hugging Face transformers pipeline is used to control the maximum number of new tokens generated by a model, ignoring the number of tokens in the input prompt.

do_sample=True: Enables sampling, where the model stochastically selects the next token from the probability distribution over the vocabulary. This introduces randomness, leading to more varied, creative, and diverse outputs. This is often used with other parameters like temperature, top_k, and top_p to control the level of randomness.

do_sample=False: Activates greedy decoding, where the model consistently picks the token with the highest probability at each step. This results in deterministic, predictable, and conservative text generation.

Finally, we create our prompt as a user and give it to the model:

In [ ]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Tell me who you think will win the world cup and which team you think has the highest chance of winning with rating."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=50000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'role': 'user', 'content': 'Tell me who you think will win the world cup and which team you think has the highest chance of winning with rating.'}, {'role': 'assistant', 'content': "As an AI, I don't have personal opinions or the ability to predict outcomes like sports events. The outcome of a soccer world cup depends on a wide range of factors, including team performance, player form, injuries, and even luck.\n\nAs of my last update in April 2023, when the 2022 FIFA World Cup took place in Qatar, the host nation, Qatar, was the winner of the tournament, defeating Argentina in the final. For the next World Cup, which would have been the 2026 FIFA World Cup, Canada, Mexico, and the United States shared hosting duties.\n\nThe favorite to win the World Cup is often considered to be the country whose national team has a strong track record of success, a high-quality squad, and is in good form. In recent years, countries like Brazil, Germany, and France have been strong contenders due to 

What is the meaning of various roles - user, assistant, system?

https://medium.com/@mudassar.hakim/mastering-prompt-engineering-a-guide-to-system-user-and-assistant-roles-in-openai-api-28fe5fbf1d81


# GPU Savings? From the Menu:
## In the top menu bar, click on Runtime.
## From the drop-down menu, select Disconnect and delete runtime.

In [ ]:
explanation = """
## What are Transformers?

Transformers are a revolutionary neural network architecture introduced in the 2017 paper "Attention Is All You Need" by Vaswani et al. They have become the backbone of most state-of-the-art models in Natural Language Processing (NLP), such as BERT, GPT, and T5.

**Key Concepts:**

1.  **Self-Attention Mechanism:** This is the core innovation. Unlike recurrent neural networks (RNNs) which process sequences token by token, self-attention allows the model to weigh the importance of different words in the input sequence when encoding each word. This enables it to capture long-range dependencies efficiently, as each word can "attend" to all other words simultaneously, regardless of their distance.

2.  **Encoder-Decoder Architecture (for some tasks):**
    *   **Encoder:** Processes the input sequence (e.g., a sentence in one language). It consists of multiple layers, each with a multi-head self-attention mechanism and a feed-forward neural network.
    *   **Decoder:** Generates the output sequence (e.g., the same sentence translated into another language). It also has multi-head self-attention (masked to prevent attending to future tokens) and a feed-forward network, but crucially, it also has an "encoder-decoder attention" layer that allows it to attend to the output of the encoder.
    *   **Encoder-only models (like BERT):** Used for tasks like classification or question answering.
    *   **Decoder-only models (like GPT):** Used for generative tasks like text generation.

3.  **Positional Encoding:** Since Transformers process words in parallel (without inherent sequential order), positional encodings are added to the input embeddings to provide information about the relative or absolute position of tokens in the sequence.

**Why are they important?**

*   **Parallelization:** Self-attention allows parallel computation, significantly speeding up training compared to sequential models like RNNs.
*   **Long-range dependencies:** They can effectively capture relationships between words that are far apart in a sentence, which was a challenge for earlier architectures.
*   **Performance:** They consistently achieve state-of-the-art results across a wide range of NLP tasks.

## Code Example with Hugging Face Transformers
"""
print(explanation)

# --- Code Example ---
# This example demonstrates a simple text generation task using a pre-trained Transformer model.
# We'll use a small model for quick demonstration. The `transformers` library should already be installed from previous cells.

from transformers import pipeline

# Load a pre-trained text generation pipeline using a lightweight model
try:
    text_generator = pipeline("text-generation", model="distilgpt2")
except Exception as e:
    print(f"Could not load distilgpt2, trying 'gpt2' as a fallback: {e}")
    text_generator = pipeline("text-generation", model="gpt2")


prompt = "The quick brown fox jumps over the lazy"
print(f"\n--- Generating text for: \"{prompt}\" ---\n")

generated_text = text_generator(prompt, max_length=50, num_return_sequences=1)

print(generated_text[0]['generated_text'])
print("\n--- End of Code Example ---")


## What are Transformers?

Transformers are a revolutionary neural network architecture introduced in the 2017 paper "Attention Is All You Need" by Vaswani et al. They have become the backbone of most state-of-the-art models in Natural Language Processing (NLP), such as BERT, GPT, and T5.

**Key Concepts:**

1.  **Self-Attention Mechanism:** This is the core innovation. Unlike recurrent neural networks (RNNs) which process sequences token by token, self-attention allows the model to weigh the importance of different words in the input sequence when encoding each word. This enables it to capture long-range dependencies efficiently, as each word can "attend" to all other words simultaneously, regardless of their distance.

2.  **Encoder-Decoder Architecture (for some tasks):**
    *   **Encoder:** Processes the input sequence (e.g., a sentence in one language). It consists of multiple layers, each with a multi-head self-attention mechanism and a feed-forward neural network.
    *   **De

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Generating text for: "The quick brown fox jumps over the lazy" ---



[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The quick brown fox jumps over the lazy hulk in the middle of the crowd.
"He's here for the rest of the game," he says.
"I'm sure he's looking forward to the time."
Riley's face is still white, and the fox was barely able to move.
'I saw her'
"She's looking forward to the time," Riley says. "She's really cute, too."
Riley walks off to the back of the house.
"I'm sure she'll be alright," she says. "She's so nice and nice and just a bit of fun."
Riley walks away.
"I'm sure she'll be alright," Riley says. "She's nice and nice and just a bit of fun."
"Riley's a good character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and she's a great character, and that's

--- End of Code Example ---
